### Setup 

In [61]:
# Import libraries
import os
import json
import requests
import io
import boto3
import pandas as pd
from dotenv import load_dotenv

In [62]:
# Load urls, passwords and access keys
load_dotenv()
ACCESS_KEY = os.getenv("AWS_KEY")
ACCESS_SECRET = os.getenv("AWS_SECRET")
upload_bucket = os.getenv("S3_BUCKET")

### Define extract_data function

In [63]:
def extract_data(url):
      try:
            response = requests.get(url, params=params)
            response.raise_for_status() # raises an exception for http errors
            return response.json()
      except requests.exceptions.RequestException as err:
            print(f"An error occured: {err}")

### Create dataframe

In [69]:
# Parse API response into pandas dataframe
def create_df():
    if len(rows) > 0:
        print ("CMS API data received")
        print(f"Downloaded {len(rows)} rows sucessfully") # print number of rows downloaded
    df = pd.DataFrame(rows)
    return df

### Load data in AWS S3

In [65]:
# Define load function
def load(filepath):
        try:
                print("Importing data in S3") 
                #
                s3_client = boto3.client('s3', aws_access_key_id=ACCESS_KEY, aws_secret_access_key=ACCESS_SECRET, region_name='us-east-1')
                with io.StringIO() as csv_buffer:
                        df.to_csv(csv_buffer, index=False)
                        response=s3_client.put_object(
                        Bucket=upload_bucket, Key="raw/"+filepath, Body=csv_buffer.getvalue()
                        )
                status = response.get("ResponseMetadata", {}).get("HTTPStatusCode")
                #
                if status == 200:
                        print("Succesful S3 put_object response")
                else:
                        print(f"Unsuccesful S3 put_object response. Status - {status}")
        except Exception as e:
                print(f"Data load error: {e}")

### Move 2026 hcahps data from CMS API to AWS S3 bucket

In [ ]:
# Set parameters
url = "https://data.cms.gov/provider-data/api/1/datastore/query/dgck-syfz/0"                 
rows = []
offset = 0
limit = 1500  
#
print("Connecting to CMS API")
# API is paginated - fetch all pages with a while loop         
while True:
      params = {
      "limit": limit,
      "offset": offset
      }    
      result = extract_data(url) # get data from API
      data = result["results"] # API response is a dictionnary of lists - we are only keeping the results list
      if not data:
            break
      rows.extend(data)
      offset += limit
#
df = create_df()
load("hcahps_data_2026.csv")

### Move cost report data from CMS API to AWS S3 bucket

In [71]:
# Set parameters
url = "https://data.cms.gov/data-api/v1/dataset/44060663-47d8-4ced-a115-b53b4c270acb/data"                 
rows = []
offset = 0
limit = 1500  
#
print("Connecting to CMS API")
# API is paginated - fetch all pages with a while loop         
while True:
      params = {
      "limit": limit,
      "offset": offset
      }    
      data = extract_data(url) # get data from API
      if not data:
            break
      rows.extend(data)
      offset += limit
#
df = create_df()
load("cost_report_2026.csv")

Connecting to CMS API
CMS API data received
Downloaded 4103 rows sucessfully
Importing data in S3
Succesful S3 put_object response
